# Proposed PEFT approach framework

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

class HResModule(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.phi_res = nn.Parameter(torch.randn(d_model, 4) * 0.02)
        self.b_res = nn.Parameter(torch.zeros(2, 2))
        self.alpha_res = nn.Parameter(torch.tensor(0.01))
    
    def sinkhorn_knopp(self, H, iterations=20):
        H = torch.exp(H)
        for _ in range(iterations):
            H = H / (H.sum(dim=1, keepdim=True) + 1e-8)
            H = H / (H.sum(dim=0, keepdim=True) + 1e-8)
        return H
    
    def forward(self, x):
        B, S, D = x.shape
        rms = torch.sqrt((x ** 2).mean(dim=-1, keepdim=True) + 1e-8)
        normed = x / rms
        h_raw = torch.matmul(normed, self.phi_res).view(B, S, 2, 2)
        h_raw = self.alpha_res * h_raw + self.b_res
        h_avg = h_raw.mean(dim=[0, 1])
        H_res = self.sinkhorn_knopp(h_avg)
        streams = x.view(B, S, 2, D // 2)
        mixed = torch.einsum('ij,bsjd->bsid', H_res, streams)
        return mixed.contiguous().view(B, S, D)

class AttentionAlignedAdapter(nn.Module):
    def __init__(self, hidden_dim=768, rank=64):
        super().__init__()
        self.Wd = nn.Linear(hidden_dim, rank, bias=False)
        self.Wu = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.kaiming_normal_(self.Wd.weight, mode='fan_out')
        nn.init.zeros_(self.Wu.weight)
    
    def forward(self, X, S):
        Z = self.Wd(X)
        Zp = torch.matmul(S, Z)
        return self.Wu(Zp)

class PEFTSelfAttention(nn.Module):
    def __init__(self, attn_self, attn_output, hidden_dim=768, rank=64):
        super().__init__()
        self.self_attn = attn_self
        self.out_dense = attn_output.dense
        self.adapter = AttentionAlignedAdapter(hidden_dim, rank)
        self.out_dense.weight.requires_grad = False
        for p in self.self_attn.parameters():
            p.requires_grad = False
    
    def forward(self, hidden_states, attention_mask):
        B, T, H = hidden_states.size()
        q = self.self_attn.query(hidden_states)
        k = self.self_attn.key(hidden_states)
        v = self.self_attn.value(hidden_states)
        num_heads = self.self_attn.num_attention_heads
        head_dim = H // num_heads
        q = q.view(B, T, num_heads, head_dim).transpose(1, 2)
        k = k.view(B, T, num_heads, head_dim).transpose(1, 2)
        v = v.view(B, T, num_heads, head_dim).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(head_dim)
        scores = scores + attention_mask
        S = F.softmax(scores, dim=-1)
        A = torch.matmul(S, v)
        A = A.transpose(1, 2).contiguous().view(B, T, H)
        deltaA = self.adapter(hidden_states, S.mean(dim=1))
        A = A + deltaA
        return self.out_dense(A)

class PEFTEncoderLayer(nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.attn = PEFTSelfAttention(layer.attention.self, layer.attention.output)
        self.attn_ln = layer.attention.output.LayerNorm
        self.ffn_int = layer.intermediate.dense
        self.ffn_out = layer.output.dense
        self.ffn_ln = layer.output.LayerNorm
        self.hres = HResModule(d_model=768)
        for p in self.attn_ln.parameters():
            p.requires_grad = False
        for p in self.ffn_int.parameters():
            p.requires_grad = False
        for p in self.ffn_out.parameters():
            p.requires_grad = False
        for p in self.ffn_ln.parameters():
            p.requires_grad = False
    
    def forward(self, x, attn_mask):
        attn_out = self.attn(x, attn_mask)
        x_mid = self.attn_ln(x + attn_out)
        enhanced_res = self.hres(x_mid)
        ff = self.ffn_out(F.gelu(self.ffn_int(x_mid)))
        x_out = self.ffn_ln(x_mid + enhanced_res + ff)
        return x_out

class PEFTUniXcoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained("microsoft/unixcoder-base")
        for p in self.model.parameters():
            p.requires_grad = False
        self.layers = nn.ModuleList([PEFTEncoderLayer(l) for l in self.model.encoder.layer])
        self.embeddings = self.model.embeddings
        self.classifier = nn.Linear(768, 2)
        nn.init.xavier_normal_(self.classifier.weight)
    
    def forward(self, input_ids, attention_mask):
        attn_mask = (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -10000.0
        x = self.embeddings(input_ids)
        for layer in self.layers:
            x = layer(x, attn_mask)
        logits = self.classifier(x[:, 0, :])
        return logits

class CodeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        func = str(self.data.iloc[idx]['code'])
        label = int(self.data.iloc[idx]['label'])
        encoding = self.tokenizer(func, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, accuracy, precision, recall, f1

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    return avg_loss, accuracy, precision, recall, f1, auc, cm

def plot_training_history(history, save_path='training_overview.png'):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Training Overview - 15 Epochs', fontsize=16, fontweight='bold')
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0, 0].plot(epochs, history['train_loss'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[0, 0].set_xlabel('Epoch', fontsize=11)
    axes[0, 0].set_ylabel('Loss', fontsize=11)
    axes[0, 0].set_title('Loss Curve', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].plot(epochs, history['train_acc'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[0, 1].plot(epochs, history['val_acc'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[0, 1].set_xlabel('Epoch', fontsize=11)
    axes[0, 1].set_ylabel('Accuracy', fontsize=11)
    axes[0, 1].set_title('Accuracy Curve', fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 2].plot(epochs, history['train_precision'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[0, 2].plot(epochs, history['val_precision'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[0, 2].set_xlabel('Epoch', fontsize=11)
    axes[0, 2].set_ylabel('Precision', fontsize=11)
    axes[0, 2].set_title('Precision Curve', fontsize=12, fontweight='bold')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    axes[1, 0].plot(epochs, history['train_recall'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[1, 0].plot(epochs, history['val_recall'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[1, 0].set_xlabel('Epoch', fontsize=11)
    axes[1, 0].set_ylabel('Recall', fontsize=11)
    axes[1, 0].set_title('Recall Curve', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 1].plot(epochs, history['train_f1'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[1, 1].plot(epochs, history['val_f1'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[1, 1].set_xlabel('Epoch', fontsize=11)
    axes[1, 1].set_ylabel('F1-Score', fontsize=11)
    axes[1, 1].set_title('F1-Score Curve', fontsize=12, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    train_final = [history['train_acc'][-1], history['train_precision'][-1], history['train_recall'][-1], history['train_f1'][-1]]
    val_final = [history['val_acc'][-1], history['val_precision'][-1], history['val_recall'][-1], history['val_f1'][-1]]
    test_final = [history['test_acc'], history['test_precision'], history['test_recall'], history['test_f1']]
    x = np.arange(len(metrics))
    width = 0.25
    axes[1, 2].bar(x - width, train_final, width, label='Train', color='#3498db')
    axes[1, 2].bar(x, val_final, width, label='Validation', color='#e74c3c')
    axes[1, 2].bar(x + width, test_final, width, label='Test', color='#2ecc71')
    axes[1, 2].set_xlabel('Metrics', fontsize=11)
    axes[1, 2].set_ylabel('Score', fontsize=11)
    axes[1, 2].set_title('Final Metrics Comparison', fontsize=12, fontweight='bold')
    axes[1, 2].set_xticks(x)
    axes[1, 2].set_xticklabels(metrics, rotation=15, ha='right')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3, axis='y')
    axes[1, 2].set_ylim([0, 1])
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\nTraining overview saved to: {save_path}")

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f'Device: {device}')
    model_name = 'microsoft/unixcoder-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    full_train_data = pd.read_csv('/traincodex.csv')
    test_data = pd.read_csv('/testcodex.csv')
    train_data, val_data = train_test_split(full_train_data, test_size=0.15, random_state=42, stratify=full_train_data['label'])
    print(f"\nDataset Split:")
    print(f"  Training samples:   {len(train_data)}")
    print(f"  Validation samples: {len(val_data)}")
    print(f"  Test samples:       {len(test_data)}")
    train_dataset = CodeDataset(train_data.reset_index(drop=True), tokenizer)
    val_dataset = CodeDataset(val_data.reset_index(drop=True), tokenizer)
    test_dataset = CodeDataset(test_data, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    train_labels = train_data['label'].values
    class_counts = np.bincount(train_labels)
    class_weights = torch.tensor([len(train_labels) / (2 * count) for count in class_counts], dtype=torch.float32).to(device)
    model = PEFTUniXcoder().to(device)
    total_params, trainable_params = count_parameters(model)
    print(f"\n{'='*80}")
    print(f"MODEL PARAMETERS")
    print(f"{'='*80}")
    print(f"Total Parameters:      {total_params:,}")
    print(f"Trainable Parameters:  {trainable_params:,}")
    print(f"Trainable Percentage:  {100 * trainable_params / total_params:.4f}%")
    print(f"{'='*80}\n")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5, weight_decay=0.01)
    from torch.optim.lr_scheduler import CosineAnnealingLR
    scheduler = CosineAnnealingLR(optimizer, T_max=15)
    history = {
        'train_loss': [], 'train_acc': [], 'train_precision': [], 'train_recall': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_precision': [], 'val_recall': [], 'val_f1': []
    }
    num_epochs = 15
    for epoch in range(num_epochs):
        print(f"\n{'='*80}")
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print(f"{'='*80}")
        train_loss, train_acc, train_prec, train_rec, train_f1 = train_epoch(model, train_loader, optimizer, criterion, device)
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, Prec: {train_prec:.4f}, Rec: {train_rec:.4f}, F1: {train_f1:.4f}")
        val_loss, val_acc, val_prec, val_rec, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, F1: {val_f1:.4f}")
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_precision'].append(train_prec)
        history['train_recall'].append(train_rec)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_precision'].append(val_prec)
        history['val_recall'].append(val_rec)
        history['val_f1'].append(val_f1)
        scheduler.step()
    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc, cm = evaluate(model, test_loader, criterion, device)
    history['test_acc'] = test_acc
    history['test_precision'] = test_prec
    history['test_recall'] = test_rec
    history['test_f1'] = test_f1
    print(f"\n{'='*80}")
    print(f"FINAL TEST RESULTS")
    print(f"{'='*80}")
    print(f"  Loss:      {test_loss:.4f}")
    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  Precision: {test_prec:.4f}")
    print(f"  Recall:    {test_rec:.4f}")
    print(f"  F1-Score:  {test_f1:.4f}")
    print(f"  AUC-ROC:   {test_auc:.4f}")
    print(f"\nConfusion Matrix:")
    print(cm)
    print(f"{'='*80}")
    plot_training_history(history)

if __name__ == "__main__":
    main()

# Ablation Study

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm
import os

class HResModule(nn.Module):
    def __init__(self, d_model=768, n_splits=2):
        super().__init__()
        self.n_splits = n_splits
        self.phi_res = nn.Parameter(torch.randn(d_model, n_splits * n_splits) * 0.02)
        self.b_res = nn.Parameter(torch.zeros(n_splits, n_splits))
        self.alpha_res = nn.Parameter(torch.tensor(0.01))
    def sinkhorn_knopp(self, H, iterations=20):
        H = torch.exp(H)
        for _ in range(iterations):
            H = H / (H.sum(dim=1, keepdim=True) + 1e-8)
            H = H / (H.sum(dim=0, keepdim=True) + 1e-8)
        return H
    def forward(self, x):
        B, S, D = x.shape
        n = self.n_splits
        rms = torch.sqrt((x ** 2).mean(dim=-1, keepdim=True) + 1e-8)
        normed = x / rms
        h_raw = torch.matmul(normed, self.phi_res).view(B, S, n, n)
        h_raw = self.alpha_res * h_raw + self.b_res
        h_avg = h_raw.mean(dim=[0, 1])
        H_res = self.sinkhorn_knopp(h_avg)
        chunk = D // n
        streams = x[:, :, :n * chunk].view(B, S, n, chunk)
        mixed = torch.einsum('ij,bsjd->bsid', H_res, streams)
        out = mixed.contiguous().view(B, S, n * chunk)
        if n * chunk < D:
            out = torch.cat([out, x[:, :, n * chunk:]], dim=-1)
        return out

class AttentionAlignedAdapter(nn.Module):
    def __init__(self, hidden_dim=768, rank=64):
        super().__init__()
        self.Wd = nn.Linear(hidden_dim, rank, bias=False)
        self.Wu = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.kaiming_normal_(self.Wd.weight, mode='fan_out')
        nn.init.zeros_(self.Wu.weight)
    def forward(self, X, S):
        Z = self.Wd(X)
        Zp = torch.matmul(S, Z)
        return self.Wu(Zp)

class PEFTSelfAttention(nn.Module):
    def __init__(self, attn_self, attn_output, hidden_dim=768, rank=64, use_adapter=True):
        super().__init__()
        self.self_attn = attn_self
        self.out_dense = attn_output.dense
        self.use_adapter = use_adapter
        if use_adapter:
            self.adapter = AttentionAlignedAdapter(hidden_dim, rank)
        self.out_dense.weight.requires_grad = False
        for p in self.self_attn.parameters():
            p.requires_grad = False
    def forward(self, hidden_states, attention_mask):
        B, T, H = hidden_states.size()
        q = self.self_attn.query(hidden_states)
        k = self.self_attn.key(hidden_states)
        v = self.self_attn.value(hidden_states)
        num_heads = self.self_attn.num_attention_heads
        head_dim = H // num_heads
        q = q.view(B, T, num_heads, head_dim).transpose(1, 2)
        k = k.view(B, T, num_heads, head_dim).transpose(1, 2)
        v = v.view(B, T, num_heads, head_dim).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(head_dim)
        scores = scores + attention_mask
        S = F.softmax(scores, dim=-1)
        A = torch.matmul(S, v)
        A = A.transpose(1, 2).contiguous().view(B, T, H)
        if self.use_adapter:
            deltaA = self.adapter(hidden_states, S.mean(dim=1))
            A = A + deltaA
        return self.out_dense(A)

class PEFTEncoderLayer(nn.Module):
    def __init__(self, layer, rank=64, use_adapter=True, hres_splits=None):
        super().__init__()
        self.attn = PEFTSelfAttention(layer.attention.self, layer.attention.output, hidden_dim=768, rank=rank, use_adapter=use_adapter)
        self.attn_ln = layer.attention.output.LayerNorm
        self.ffn_int = layer.intermediate.dense
        self.ffn_out = layer.output.dense
        self.ffn_ln = layer.output.LayerNorm
        self.hres = HResModule(d_model=768, n_splits=hres_splits) if hres_splits is not None else None
        for p in self.attn_ln.parameters():
            p.requires_grad = False
        for p in self.ffn_int.parameters():
            p.requires_grad = False
        for p in self.ffn_out.parameters():
            p.requires_grad = False
        for p in self.ffn_ln.parameters():
            p.requires_grad = False
    def forward(self, x, attn_mask):
        attn_out = self.attn(x, attn_mask)
        x_mid = self.attn_ln(x + attn_out)
        if self.hres is not None:
            enhanced_res = self.hres(x_mid)
            ff = self.ffn_out(F.gelu(self.ffn_int(x_mid)))
            x_out = self.ffn_ln(x_mid + enhanced_res + ff)
        else:
            ff = self.ffn_out(F.gelu(self.ffn_int(x_mid)))
            x_out = self.ffn_ln(x_mid + ff)
        return x_out

class PEFTUniXcoder(nn.Module):
    def __init__(self, rank=64, use_adapter=True, hres_splits=None):
        super().__init__()
        self.model = AutoModel.from_pretrained("microsoft/unixcoder-base")
        for p in self.model.parameters():
            p.requires_grad = False
        self.layers = nn.ModuleList([PEFTEncoderLayer(l, rank=rank, use_adapter=use_adapter, hres_splits=hres_splits) for l in self.model.encoder.layer])
        self.embeddings = self.model.embeddings
        self.classifier = nn.Linear(768, 2)
        nn.init.xavier_normal_(self.classifier.weight)
    def forward(self, input_ids, attention_mask):
        attn_mask = (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -10000.0
        x = self.embeddings(input_ids)
        for layer in self.layers:
            x = layer(x, attn_mask)
        logits = self.classifier(x[:, 0, :])
        return logits

class FrozenUniXcoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained("microsoft/unixcoder-base")
        for p in self.model.parameters():
            p.requires_grad = False
        self.classifier = nn.Linear(768, 2)
        nn.init.xavier_normal_(self.classifier.weight)
    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.classifier(outputs.last_hidden_state[:, 0, :])
        return logits

class CodeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512, code_col='code'):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.code_col = code_col
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        func = str(self.data.iloc[idx][self.code_col])
        label = int(self.data.iloc[idx]['label'])
        encoding = self.tokenizer(func, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': encoding['input_ids'].squeeze(0), 'attention_mask': encoding['attention_mask'].squeeze(0), 'label': torch.tensor(label, dtype=torch.long)}

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, accuracy, precision, recall, f1

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    return avg_loss, accuracy, precision, recall, f1, auc, cm

def run_ablation_config(config_name, model, train_loader, val_loader, test_loader, criterion, device, num_epochs=15):
    print(f"\n{'='*80}")
    print(f"ABLATION CONFIG: {config_name}")
    print(f"{'='*80}")
    total_params, trainable_params = count_parameters(model)
    print(f"Total Parameters:     {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Trainable Percentage: {100 * trainable_params / total_params:.4f}%")
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5, weight_decay=0.01)
    from torch.optim.lr_scheduler import CosineAnnealingLR
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    best_val_f1 = -1.0
    best_state = None
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        train_loss, train_acc, train_prec, train_rec, train_f1 = train_epoch(model, train_loader, optimizer, criterion, device)
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, Prec: {train_prec:.4f}, Rec: {train_rec:.4f}, F1: {train_f1:.4f}")
        val_loss, val_acc, val_prec, val_rec, val_f1, val_auc, _ = evaluate(model, val_loader, criterion, device)
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  [*] Best model updated at epoch {epoch + 1} with Val F1 = {best_val_f1:.4f}")
        scheduler.step()
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc, test_cm = evaluate(model, test_loader, criterion, device)
    print(f"\n{'='*80}")
    print(f"TEST RESULTS — {config_name}")
    print(f"{'='*80}")
    print(f"  Loss:      {test_loss:.4f}")
    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  Precision: {test_prec:.4f}")
    print(f"  Recall:    {test_rec:.4f}")
    print(f"  F1-Score:  {test_f1:.4f}")
    print(f"  AUC-ROC:   {test_auc:.4f}")
    print(f"\nConfusion Matrix:")
    print(test_cm)
    return {'config': config_name, 'loss': test_loss, 'accuracy': test_acc, 'precision': test_prec, 'recall': test_rec, 'f1': test_f1, 'auc': test_auc, 'cm': test_cm}

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f'Device: {device}')
    model_name = 'microsoft/unixcoder-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    full_train_data = pd.read_csv('traincodex.csv')
    test_data = pd.read_csv('testcodex.csv')
    train_data, val_data = train_test_split(full_train_data, test_size=0.15, random_state=42, stratify=full_train_data['label'])
    print(f"\nDataset Split:")
    print(f"  Training samples:   {len(train_data)}")
    print(f"  Validation samples: {len(val_data)}")
    print(f"  Test samples:       {len(test_data)}")
    train_dataset = CodeDataset(train_data.reset_index(drop=True), tokenizer, code_col='code')
    val_dataset = CodeDataset(val_data.reset_index(drop=True), tokenizer, code_col='code')
    test_dataset = CodeDataset(test_data.reset_index(drop=True), tokenizer, code_col='code')
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    train_labels = train_data['label'].values
    class_counts = np.bincount(train_labels)
    class_weights = torch.tensor([len(train_labels) / (2 * count) for count in class_counts], dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    ablation_configs = [
        {'name': 'Full_Proposed_Rank64_HRes2', 'use_adapter': True, 'rank': 64, 'hres_splits': 2, 'frozen_only': False},
        {'name': 'Rank64_HRes1', 'use_adapter': True, 'rank': 64, 'hres_splits': 1, 'frozen_only': False},
        {'name': 'Rank64_HRes3', 'use_adapter': True, 'rank': 64, 'hres_splits': 3, 'frozen_only': False},
        {'name': 'Rank64_HRes4', 'use_adapter': True, 'rank': 64, 'hres_splits': 4, 'frozen_only': False},
        {'name': 'Rank64_NoHRes', 'use_adapter': True, 'rank': 64, 'hres_splits': None, 'frozen_only': False},
        {'name': 'NoRank_HRes2_Only', 'use_adapter': False, 'rank': 64, 'hres_splits': 2, 'frozen_only': False},
        {'name': 'FrozenUniXcoder_ClassifierOnly', 'use_adapter': False, 'rank': 64, 'hres_splits': None, 'frozen_only': True},
    ]
    all_results = []
    for cfg in ablation_configs:
        if cfg['frozen_only']:
            model = FrozenUniXcoder().to(device)
        else:
            model = PEFTUniXcoder(rank=cfg['rank'], use_adapter=cfg['use_adapter'], hres_splits=cfg['hres_splits']).to(device)
        result = run_ablation_config(cfg['name'], model, train_loader, val_loader, test_loader, criterion, device, num_epochs=15)
        all_results.append(result)
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f"\n{'='*80}")
    print("ABLATION STUDY — FINAL SUMMARY")
    print(f"{'='*80}")
    header = f"{'Config':<40} {'Loss':>8} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'AUC':>8}"
    print(header)
    print('-' * len(header))
    for r in all_results:
        print(f"{r['config']:<40} {r['loss']:>8.4f} {r['accuracy']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f} {r['f1']:>8.4f} {r['auc']:>8.4f}")
    print(f"{'='*80}")

if __name__ == "__main__":
    main()

# Different bentchmark analysis with saved modle

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import os


class HResModule(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.phi_res = nn.Parameter(torch.randn(d_model, 4) * 0.02)
        self.b_res = nn.Parameter(torch.zeros(2, 2))
        self.alpha_res = nn.Parameter(torch.tensor(0.01))

    def sinkhorn_knopp(self, H, iterations=20):
        H = torch.exp(H)
        for _ in range(iterations):
            H = H / (H.sum(dim=1, keepdim=True) + 1e-8)
            H = H / (H.sum(dim=0, keepdim=True) + 1e-8)
        return H

    def forward(self, x):
        B, S, D = x.shape
        rms = torch.sqrt((x ** 2).mean(dim=-1, keepdim=True) + 1e-8)
        normed = x / rms
        h_raw = torch.matmul(normed, self.phi_res).view(B, S, 2, 2)
        h_raw = self.alpha_res * h_raw + self.b_res
        h_avg = h_raw.mean(dim=[0, 1])
        H_res = self.sinkhorn_knopp(h_avg)
        streams = x.view(B, S, 2, D // 2)
        mixed = torch.einsum('ij,bsjd->bsid', H_res, streams)
        return mixed.contiguous().view(B, S, D)


class AttentionAlignedAdapter(nn.Module):
    def __init__(self, hidden_dim=768, rank=64):
        super().__init__()
        self.Wd = nn.Linear(hidden_dim, rank, bias=False)
        self.Wu = nn.Linear(rank, hidden_dim, bias=False)
        nn.init.kaiming_normal_(self.Wd.weight, mode='fan_out')
        nn.init.zeros_(self.Wu.weight)

    def forward(self, X, S):
        Z = self.Wd(X)
        Zp = torch.matmul(S, Z)
        return self.Wu(Zp)


class PEFTSelfAttention(nn.Module):
    def __init__(self, attn_self, attn_output, hidden_dim=768, rank=64):
        super().__init__()
        self.self_attn = attn_self
        self.out_dense = attn_output.dense
        self.adapter = AttentionAlignedAdapter(hidden_dim, rank)
        self.out_dense.weight.requires_grad = False
        for p in self.self_attn.parameters():
            p.requires_grad = False

    def forward(self, hidden_states, attention_mask):
        B, T, H = hidden_states.size()
        q = self.self_attn.query(hidden_states)
        k = self.self_attn.key(hidden_states)
        v = self.self_attn.value(hidden_states)
        num_heads = self.self_attn.num_attention_heads
        head_dim = H // num_heads
        q = q.view(B, T, num_heads, head_dim).transpose(1, 2)
        k = k.view(B, T, num_heads, head_dim).transpose(1, 2)
        v = v.view(B, T, num_heads, head_dim).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(head_dim)
        scores = scores + attention_mask
        S = F.softmax(scores, dim=-1)
        A = torch.matmul(S, v)
        A = A.transpose(1, 2).contiguous().view(B, T, H)
        deltaA = self.adapter(hidden_states, S.mean(dim=1))
        A = A + deltaA
        return self.out_dense(A)


class PEFTEncoderLayer(nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.attn = PEFTSelfAttention(layer.attention.self, layer.attention.output)
        self.attn_ln = layer.attention.output.LayerNorm
        self.ffn_int = layer.intermediate.dense
        self.ffn_out = layer.output.dense
        self.ffn_ln = layer.output.LayerNorm
        self.hres = HResModule(d_model=768)
        for p in self.attn_ln.parameters():
            p.requires_grad = False
        for p in self.ffn_int.parameters():
            p.requires_grad = False
        for p in self.ffn_out.parameters():
            p.requires_grad = False
        for p in self.ffn_ln.parameters():
            p.requires_grad = False

    def forward(self, x, attn_mask):
        attn_out = self.attn(x, attn_mask)
        x_mid = self.attn_ln(x + attn_out)
        enhanced_res = self.hres(x_mid)
        ff = self.ffn_out(F.gelu(self.ffn_int(x_mid)))
        x_out = self.ffn_ln(x_mid + enhanced_res + ff)
        return x_out


class PEFTUniXcoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained("microsoft/unixcoder-base")
        for p in self.model.parameters():
            p.requires_grad = False
        self.layers = nn.ModuleList([PEFTEncoderLayer(l) for l in self.model.encoder.layer])
        self.embeddings = self.model.embeddings
        self.classifier = nn.Linear(768, 2)
        nn.init.xavier_normal_(self.classifier.weight)

    def forward(self, input_ids, attention_mask):
        attn_mask = (1.0 - attention_mask.unsqueeze(1).unsqueeze(2)) * -10000.0
        x = self.embeddings(input_ids)
        for layer in self.layers:
            x = layer(x, attn_mask)
        logits = self.classifier(x[:, 0, :])
        return logits


class CodeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512, code_col='code'):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.code_col = code_col

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        func = str(self.data.iloc[idx][self.code_col])
        label = int(self.data.iloc[idx]['label'])
        encoding = self.tokenizer(func, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params


def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return avg_loss, accuracy, precision, recall, f1


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)
    cm = confusion_matrix(all_labels, all_preds)
    return avg_loss, accuracy, precision, recall, f1, auc, cm


def plot_training_history(history, save_path='training_overview.png'):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Training Overview - 15 Epochs', fontsize=16, fontweight='bold')
    epochs = range(1, len(history['train_loss']) + 1)

    axes[0, 0].plot(epochs, history['train_loss'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[0, 0].set_xlabel('Epoch', fontsize=11)
    axes[0, 0].set_ylabel('Loss', fontsize=11)
    axes[0, 0].set_title('Loss Curve', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(epochs, history['train_acc'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[0, 1].plot(epochs, history['val_acc'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[0, 1].set_xlabel('Epoch', fontsize=11)
    axes[0, 1].set_ylabel('Accuracy', fontsize=11)
    axes[0, 1].set_title('Accuracy Curve', fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[0, 2].plot(epochs, history['train_precision'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[0, 2].plot(epochs, history['val_precision'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[0, 2].set_xlabel('Epoch', fontsize=11)
    axes[0, 2].set_ylabel('Precision', fontsize=11)
    axes[0, 2].set_title('Precision Curve', fontsize=12, fontweight='bold')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    axes[1, 0].plot(epochs, history['train_recall'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[1, 0].plot(epochs, history['val_recall'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[1, 0].set_xlabel('Epoch', fontsize=11)
    axes[1, 0].set_ylabel('Recall', fontsize=11)
    axes[1, 0].set_title('Recall Curve', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(epochs, history['train_f1'], 'b-o', label='Train', linewidth=2, markersize=6)
    axes[1, 1].plot(epochs, history['val_f1'], 'r-s', label='Validation', linewidth=2, markersize=6)
    axes[1, 1].set_xlabel('Epoch', fontsize=11)
    axes[1, 1].set_ylabel('F1-Score', fontsize=11)
    axes[1, 1].set_title('F1-Score Curve', fontsize=12, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    train_final = [history['train_acc'][-1], history['train_precision'][-1], history['train_recall'][-1], history['train_f1'][-1]]
    val_final = [history['val_acc'][-1], history['val_precision'][-1], history['val_recall'][-1], history['val_f1'][-1]]
    test_final = [history['test_acc'], history['test_precision'], history['test_recall'], history['test_f1']]
    x = np.arange(len(metrics))
    width = 0.25
    axes[1, 2].bar(x - width, train_final, width, label='Train', color='#3498db')
    axes[1, 2].bar(x, val_final, width, label='Validation', color='#e74c3c')
    axes[1, 2].bar(x + width, test_final, width, label='Test', color='#2ecc71')
    axes[1, 2].set_xlabel('Metrics', fontsize=11)
    axes[1, 2].set_ylabel('Score', fontsize=11)
    axes[1, 2].set_title('Final Metrics Comparison', fontsize=12, fontweight='bold')
    axes[1, 2].set_xticks(x)
    axes[1, 2].set_xticklabels(metrics, rotation=15, ha='right')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3, axis='y')
    axes[1, 2].set_ylim([0, 1])
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Training overview saved to: {save_path}")


def plot_inference_results(all_results, save_dir='.'):
    dataset_names = [r['name'] for r in all_results]
    accuracies   = [r['accuracy']  for r in all_results]
    precisions   = [r['precision'] for r in all_results]
    recalls      = [r['recall']    for r in all_results]
    f1s          = [r['f1']        for r in all_results]
    aucs         = [r['auc']       for r in all_results]

    palette = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#44BBA4', '#E94F37', '#393E41']

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('Cross-Dataset Inference Performance — PEFTUniXcoder', fontsize=17, fontweight='bold', y=1.01)

    x = np.arange(len(dataset_names))
    width = 0.6

    metric_data = [
        (accuracies,  'Accuracy',  axes[0, 0]),
        (precisions,  'Precision', axes[0, 1]),
        (recalls,     'Recall',    axes[0, 2]),
        (f1s,         'F1-Score',  axes[1, 0]),
        (aucs,        'AUC-ROC',   axes[1, 1]),
    ]

    for values, metric_name, ax in metric_data:
        bars = ax.bar(x, values, width=width, color=palette[:len(dataset_names)], edgecolor='white', linewidth=0.8, zorder=3)
        ax.set_xticks(x)
        ax.set_xticklabels(dataset_names, rotation=35, ha='right', fontsize=9)
        ax.set_ylabel(metric_name, fontsize=11)
        ax.set_title(f'{metric_name} per Dataset', fontsize=12, fontweight='bold')
        ax.set_ylim([0, 1.08])
        ax.grid(True, alpha=0.3, axis='y', zorder=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() + 0.012,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    metric_matrix = np.array([accuracies, precisions, recalls, f1s, aucs])
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
    cmap = LinearSegmentedColormap.from_list('perf', ['#fff7ec', '#fc8d59', '#d73027'])
    im = axes[1, 2].imshow(metric_matrix, aspect='auto', cmap=cmap, vmin=0, vmax=1)
    axes[1, 2].set_xticks(range(len(dataset_names)))
    axes[1, 2].set_xticklabels(dataset_names, rotation=35, ha='right', fontsize=9)
    axes[1, 2].set_yticks(range(len(metric_labels)))
    axes[1, 2].set_yticklabels(metric_labels, fontsize=10)
    axes[1, 2].set_title('Performance Heatmap', fontsize=12, fontweight='bold')
    for i in range(len(metric_labels)):
        for j in range(len(dataset_names)):
            axes[1, 2].text(j, i, f'{metric_matrix[i, j]:.3f}', ha='center', va='center', fontsize=8, fontweight='bold',
                            color='black' if metric_matrix[i, j] > 0.5 else 'white')
    plt.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    save_path = os.path.join(save_dir, 'inference_cross_dataset_results.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Inference cross-dataset figure saved to: {save_path}")


def plot_confusion_matrices(all_results, save_dir='.'):
    n = len(all_results)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 4))
    fig.suptitle('Confusion Matrices — Cross-Dataset Inference', fontsize=16, fontweight='bold', y=1.01)
    axes = axes.flatten()
    for i, result in enumerate(all_results):
        cm = result['cm']
        sns.heatmap(cm, annot=True, fmt='d', ax=axes[i],
                    cmap='Blues', linewidths=0.5,
                    xticklabels=['Predicted 0', 'Predicted 1'],
                    yticklabels=['Actual 0', 'Actual 1'],
                    cbar=False)
        axes[i].set_title(result['name'], fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Predicted', fontsize=9)
        axes[i].set_ylabel('Actual', fontsize=9)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    save_path = os.path.join(save_dir, 'inference_confusion_matrices.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrices figure saved to: {save_path}")


def plot_radar_chart(all_results, save_dir='.'):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
    keys    = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    N = len(metrics)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    palette = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B', '#44BBA4', '#E94F37', '#393E41']
    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, size=12)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=8)

    for idx, result in enumerate(all_results):
        values = [result[k] for k in keys]
        values += values[:1]
        color = palette[idx % len(palette)]
        ax.plot(angles, values, 'o-', linewidth=2, label=result['name'], color=color)
        ax.fill(angles, values, alpha=0.08, color=color)

    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
    ax.set_title('Radar Chart — Cross-Dataset Performance', size=14, fontweight='bold', pad=20)
    plt.tight_layout()
    save_path = os.path.join(save_dir, 'inference_radar_chart.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to: {save_path}")


def plot_summary_table(all_results, save_dir='.'):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
    keys    = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    rows    = [[r['name']] + [f"{r[k]:.4f}" for k in keys] for r in all_results]
    col_labels = ['Dataset'] + metrics

    fig, ax = plt.subplots(figsize=(14, 0.55 * len(rows) + 2))
    ax.axis('off')
    table = ax.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.6)

    header_color = '#2E4057'
    for j in range(len(col_labels)):
        table[0, j].set_facecolor(header_color)
        table[0, j].set_text_props(color='white', fontweight='bold')

    for i in range(1, len(rows) + 1):
        row_color = '#f5f5f5' if i % 2 == 0 else 'white'
        for j in range(len(col_labels)):
            table[i, j].set_facecolor(row_color)

    ax.set_title('Cross-Dataset Inference Summary Table', fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    save_path = os.path.join(save_dir, 'inference_summary_table.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Summary table figure saved to: {save_path}")


def run_inference_on_dataset(model, tokenizer, csv_path, label_col, code_col, name, device, batch_size=8):
    print(f"\n  Running inference on: {name}")
    df = pd.read_csv(csv_path)
    df = df.rename(columns={label_col: 'label', code_col: 'code_col_internal'})
    dataset = CodeDataset(df.reset_index(drop=True), tokenizer, code_col='code_col_internal')
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"  {name}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            logits = model(input_ids, attention_mask)
            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
    auc  = roc_auc_score(all_labels, all_probs)
    cm   = confusion_matrix(all_labels, all_preds)

    print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f}")
    return {
        'name': name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc': auc,
        'cm': cm
    }


def main():
    save_dir = '.'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f'Device: {device}')

    model_name = 'microsoft/unixcoder-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    full_train_data = pd.read_csv('traincodex.csv')
    test_data       = pd.read_csv('/testcodex.csv')

    train_data, val_data = train_test_split(
        full_train_data, test_size=0.15, random_state=42, stratify=full_train_data['label']
    )

    print(f"\nDataset Split:")
    print(f"  Training samples:   {len(train_data)}")
    print(f"  Validation samples: {len(val_data)}")
    print(f"  Test samples:       {len(test_data)}")

    train_dataset = CodeDataset(train_data.reset_index(drop=True), tokenizer, code_col='code')
    val_dataset   = CodeDataset(val_data.reset_index(drop=True),   tokenizer, code_col='code')
    test_dataset  = CodeDataset(test_data.reset_index(drop=True),  tokenizer, code_col='code')

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False)

    train_labels  = train_data['label'].values
    class_counts  = np.bincount(train_labels)
    class_weights = torch.tensor(
        [len(train_labels) / (2 * count) for count in class_counts], dtype=torch.float32
    ).to(device)

    model = PEFTUniXcoder().to(device)

    total_params, trainable_params = count_parameters(model)
    print(f"\n{'='*80}")
    print(f"MODEL PARAMETERS")
    print(f"{'='*80}")
    print(f"Total Parameters:      {total_params:,}")
    print(f"Trainable Parameters:  {trainable_params:,}")
    print(f"Trainable Percentage:  {100 * trainable_params / total_params:.4f}%")
    print(f"{'='*80}\n")

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5, weight_decay=0.01
    )
    from torch.optim.lr_scheduler import CosineAnnealingLR
    scheduler = CosineAnnealingLR(optimizer, T_max=15)

    history = {
        'train_loss': [], 'train_acc': [], 'train_precision': [], 'train_recall': [], 'train_f1': [],
        'val_loss':   [], 'val_acc':   [], 'val_precision':   [], 'val_recall':   [], 'val_f1':   []
    }

    num_epochs   = 15
    best_val_f1  = -1.0
    best_model_path = os.path.join(save_dir, 'best_peft_unixcoder.pt')

    for epoch in range(num_epochs):
        print(f"\n{'='*80}")
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print(f"{'='*80}")

        train_loss, train_acc, train_prec, train_rec, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, Prec: {train_prec:.4f}, Rec: {train_rec:.4f}, F1: {train_f1:.4f}")

        val_loss, val_acc, val_prec, val_rec, val_f1, val_auc, _ = evaluate(
            model, val_loader, criterion, device
        )
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_precision'].append(train_prec)
        history['train_recall'].append(train_rec)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_precision'].append(val_prec)
        history['val_recall'].append(val_rec)
        history['val_f1'].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"  [*] Best model saved at epoch {epoch + 1} with Val F1 = {best_val_f1:.4f}")

        scheduler.step()

    print(f"\nLoading best saved model from: {best_model_path}")
    model.load_state_dict(torch.load(best_model_path, map_location=device))

    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc, test_cm = evaluate(
        model, test_loader, criterion, device
    )
    history['test_acc']       = test_acc
    history['test_precision'] = test_prec
    history['test_recall']    = test_rec
    history['test_f1']        = test_f1

    print(f"\n{'='*80}")
    print(f"FINAL TEST RESULTS (CDL_TEST — best model)")
    print(f"{'='*80}")
    print(f"  Loss:      {test_loss:.4f}")
    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  Precision: {test_prec:.4f}")
    print(f"  Recall:    {test_rec:.4f}")
    print(f"  F1-Score:  {test_f1:.4f}")
    print(f"  AUC-ROC:   {test_auc:.4f}")
    print(f"\nConfusion Matrix:")
    print(test_cm)
    print(f"{'='*80}")

    plot_training_history(history, save_path=os.path.join(save_dir, 'training_overview.png'))

    test_configs = [
        {
            'csv': '/big_vuln.csv',
            'label_col': 'label',
            'code_col': 'func',
            'name': 'BIG_VULTEST'
        },
        {
            'csv': '/t/mix_vuln.csv',
            'label_col': 'label',
            'code_col': 'func',
            'name': 'MIX_VULTEST'
        },
        {
            'csv': '/revealn.csv',
            'label_col': 'label',
            'code_col': 'func',
            'name': 'REVEAL_VULTEST'
        },
        {
            'csv': '/julietn.csv',
            'label_col': 'label',
            'code_col': 'func',
            'name': 'JULIET_VULTEST'
        },
        {
            'csv': '/diversen.csv',
            'label_col': 'label',
            'code_col': 'func',
            'name': 'DIVERSE_VULTEST'
        },
        {
            'csv': '/sven.csv',
            'label_col': 'label',
            'code_col': 'func',
            'name': 'SVEN'
        },
        {
            'csv': '/primetestn.csv',
            'label_col': 'label',
            'code_col': 'func',
            'name': 'Prime'
        },
        {
            'csv': '/testcodex.csv',
            'label_col': 'label',
            'code_col': 'code',
            'name': 'CDL_TEST'
        }
    ]

    print(f"\n{'='*80}")
    print("CROSS-DATASET INFERENCE EVALUATION")
    print(f"{'='*80}")

    all_results = []
    for cfg in test_configs:
        result = run_inference_on_dataset(
            model=model,
            tokenizer=tokenizer,
            csv_path=cfg['csv'],
            label_col=cfg['label_col'],
            code_col=cfg['code_col'],
            name=cfg['name'],
            device=device,
            batch_size=8
        )
        all_results.append(result)

    plot_inference_results(all_results,  save_dir=save_dir)
    plot_confusion_matrices(all_results, save_dir=save_dir)
    plot_radar_chart(all_results,        save_dir=save_dir)
    plot_summary_table(all_results,      save_dir=save_dir)

    print(f"\n{'='*80}")
    print("ALL OUTPUTS SAVED")
    print(f"  best_peft_unixcoder.pt")
    print(f"  training_overview.png")
    print(f"  inference_cross_dataset_results.png")
    print(f"  inference_confusion_matrices.png")
    print(f"  inference_radar_chart.png")
    print(f"  inference_summary_table.png")
    print(f"{'='*80}")


if __name__ == "__main__":
    main()